# 003 · INI Configuration with configparser

> **Standard Library › Logging & Config** · Python 3.14

`configparser` reads and writes INI-style configuration files — `[sections]` of `key = value` pairs. It's simple, human-editable, and built in, which is why tools like `setup.cfg`, `tox.ini`, `pytest.ini`, `.gitconfig`-style files and `logging.config.fileConfig` use it.

**In this notebook you will learn:**

- how INI files and `ConfigParser` compare to `IConfiguration` and `appsettings.json`
- reading config from strings and files; sections and keys
- typed access with `getint`/`getfloat`/`getboolean` and `fallback=`
- the `DEFAULT` section and value interpolation (basic and extended)
- writing configuration back out
- when to prefer TOML or environment variables instead

## .NET mental model

| .NET (`Microsoft.Extensions.Configuration`) | Python `configparser` |
|---|---|
| `appsettings.json` | an `.ini` / `.cfg` file |
| `config.GetSection("Database")` | `config["database"]` (a `SectionProxy`) |
| `config["Database:Host"]` | `config["database"]["host"]` |
| `config.GetValue<int>("Database:Port", 5432)` | `config.getint("database", "port", fallback=5432)` |
| `AddIniFile("app.ini")` (`Microsoft.Extensions.Configuration.Ini`) | `config.read("app.ini")` |
| Layered providers (later wins) | `config.read([...])` with several files (later wins) |
| Options binding (`IOptions<T>`) | no built-in binding — map to a `dataclass` yourself |

Important differences: INI has **only two levels** (section → key), **every value is a string** until you ask for a type, and keys are **case-insensitive** (lower-cased).

## Reading configuration

For demos we use `read_string`; in an app you'd use `read(path)` or `read_file(f)`. Lines starting with `#` or `;` are comments.

In [ ]:
import configparser

INI = """
# Sample application settings
[server]
host = 127.0.0.1
port = 8080
debug = yes
timeout_seconds = 2.5

[database]
Host = db.internal.example
Port = 5432
name = shop
"""

config = configparser.ConfigParser()
config.read_string(INI)

print(config.sections())                     # ['server', 'database'] — DEFAULT is not listed
print(config["server"]["host"])              # dictionary-style access
print(config.get("database", "name"))        # method-style access
print("database" in config, "cache" in config)

A section behaves like a dictionary of **strings**:

In [ ]:
server = config["server"]
print(type(server).__name__, dict(server))
print(repr(server["port"]))                  # '8080' — a str, not an int!
print(config["database"]["HOST"])            # keys are case-insensitive (lower-cased on read)
print(list(config["database"]))              # ['host', 'port', 'name']

## Typed getters and fallbacks

| Method | Returns |
|---|---|
| `getint(section, key)` | `int` |
| `getfloat(section, key)` | `float` |
| `getboolean(section, key)` | `bool` — accepts `1/0`, `yes/no`, `true/false`, `on/off` (case-insensitive) |
| `get(section, key)` | `str` |

All accept `fallback=` — the default if the **section or key** is missing (like `GetValue<T>(key, defaultValue)`). The same methods exist on a section proxy without the section argument.

```csharp
// C#
int port      = config.GetValue<int>("Server:Port");
bool debug    = config.GetValue<bool>("Server:Debug");
int retries   = config.GetValue("Server:Retries", 3);
```

In [ ]:
port = config.getint("server", "port")
debug = config.getboolean("server", "debug")         # "yes" -> True
timeout = config.getfloat("server", "timeout_seconds")
retries = config.getint("server", "retries", fallback=3)
cache_ttl = config["cache"].getint("ttl", 60) if "cache" in config else 60

print(port, debug, timeout, retries, cache_ttl)
print(config["server"].getint("port") + 1)           # section-proxy style

⚠️ **Gotcha:** without `fallback=`, missing entries raise — and bad values raise `ValueError` even when you pass a fallback (the fallback is only for *missing* keys).

In [ ]:
attempts = [
    lambda: config.get("server", "retries"),
    lambda: config.get("cache", "ttl"),
    lambda: config["cache"],
    lambda: config.getint("database", "host", fallback=0),
]
for attempt in attempts:
    try:
        attempt()
    except (configparser.Error, KeyError, ValueError) as e:
        print(f"{type(e).__name__}: {e}")

**Tip:** turn config into a typed object once, at start-up, rather than calling `getint` throughout your code — the Python take on `IOptions<T>` *(see Syntax › Classes & OOP › 008 · Dataclasses)*.

In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class ServerSettings:
    host: str
    port: int
    debug: bool = False

    @classmethod
    def from_section(cls, s: configparser.SectionProxy) -> "ServerSettings":
        return cls(host=s.get("host"), port=s.getint("port"), debug=s.getboolean("debug", False))

print(ServerSettings.from_section(config["server"]))

## The `DEFAULT` section

Keys in `[DEFAULT]` are visible in **every** section unless overridden — a simple form of shared settings.

In [ ]:
config = configparser.ConfigParser()
config.read_string("""
[DEFAULT]
log_level = INFO
retries = 3

[worker]
retries = 5

[scheduler]
""")

print(config["worker"]["retries"], config["worker"]["log_level"])     # 5 INFO
print(config["scheduler"]["retries"])                                 # 3 (inherited)
print(dict(config["scheduler"]))
print(config.defaults())

## Interpolation

By default (`BasicInterpolation`), a value can reference **other keys in the same section (or DEFAULT)** with `%(name)s`:

In [ ]:
config = configparser.ConfigParser()
config.read_string("""
[paths]
base = /srv/app
logs = %(base)s/logs
data = %(base)s/data
discount = 15%%
""")

print(config["paths"]["logs"])          # /srv/app/logs
print(config["paths"]["discount"])      # 15%  — a literal % must be written as %%
print(config.get("paths", "logs", raw=True))   # the un-interpolated text

⚠️ **Gotcha:** because `%` is special, values like passwords or URL-encoded strings containing `%` break with basic interpolation. Use `ConfigParser(interpolation=None)` (or `RawConfigParser`) when you don't need interpolation.

In [ ]:
broken = configparser.ConfigParser()
broken.read_string("[api]\ntoken = abc%xyz\n")      # obviously fake value
try:
    broken["api"]["token"]
except configparser.InterpolationSyntaxError as e:
    print(f"{type(e).__name__}: {e}")

raw = configparser.ConfigParser(interpolation=None)
raw.read_string("[api]\ntoken = abc%xyz\n")
print(raw["api"]["token"])

`ExtendedInterpolation` uses `${key}` (same section) and `${section:key}` (other sections) — closer to configuration placeholders you may know from other tools:

In [ ]:
config = configparser.ConfigParser(interpolation=configparser.ExtendedInterpolation())
config.read_string("""
[common]
root = /srv/shop

[web]
static = ${common:root}/static
port = 8080
url = http://localhost:${port}/
""")

print(config["web"]["static"])     # /srv/shop/static
print(config["web"]["url"])        # http://localhost:8080/

## Reading files (and layering)

`read()` accepts one path or a list. Files are applied in order, later values overriding earlier ones — like adding `appsettings.json` then `appsettings.Development.json`. It returns the list of files it actually read.

⚠️ **Gotcha:** `read()` **silently skips missing files**. If a file is mandatory, check the return value or open it yourself and use `read_file(f)`, which raises if the file doesn't exist.

In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as d:
    base_ini = Path(d) / "app.ini"
    dev_ini = Path(d) / "app.dev.ini"
    base_ini.write_text("[server]\nport = 80\ndebug = no\n", encoding="utf-8")
    dev_ini.write_text("[server]\ndebug = yes\n", encoding="utf-8")

    config = configparser.ConfigParser()
    loaded = config.read([base_ini, dev_ini, Path(d) / "app.local.ini"], encoding="utf-8")
    print([Path(p).name for p in loaded])              # the missing file is just skipped
    print(config.getint("server", "port"), config.getboolean("server", "debug"))   # 80 True

## Writing configuration

Build or modify a `ConfigParser` like a dictionary of dictionaries, then `write()` it to any text file object. Values must be strings when set via `set()`; dict-style assignment converts for you on section creation.

In [ ]:
import io

out = configparser.ConfigParser()
out["server"] = {"host": "0.0.0.0", "port": "9000"}
out["features"] = {}
out["features"]["dark_mode"] = "on"
out.set("server", "workers", str(4))          # set() requires a str

buf = io.StringIO()
out.write(buf)
print(buf.getvalue())

⚠️ **Gotcha:** `write()` does **not** preserve comments or formatting from a file you read earlier — round-tripping a hand-edited file loses the comments. Writing to a real file: `with open(path, "w", encoding="utf-8") as f: config.write(f)`.

## Customising the parser

A few useful constructor options:

- `inline_comment_prefixes=("#", ";")` — allow `key = value  # comment` (off by default; otherwise the comment becomes part of the value)
- `converters={"list": ...}` — adds your own typed getter, e.g. `getlist`
- `allow_no_value=True` — keys without values (`[flags]\nverbose`)

In [ ]:
parser = configparser.ConfigParser(
    inline_comment_prefixes=("#",),
    converters={"list": lambda v: [item.strip() for item in v.split(",") if item.strip()]},
)
parser.read_string("""
[cors]
origins = http://localhost:3000, http://localhost:5173   # dev front-ends
max_age = 600 # seconds
""")
print(parser.getlist("cors", "origins"))
print(parser.getint("cors", "max_age"))

## Modern alternatives: TOML and environment variables

INI has no nesting, no lists, and no real types. For new projects consider:

- **TOML** — typed values, arrays and nested tables; read with the built-in `tomllib` (it's what `pyproject.toml` uses) *(see Standard Library › Data Formats › 003 · TOML with tomllib)*.
- **JSON** — if your settings are already JSON, like `appsettings.json` *(see Standard Library › Data Formats › 001 · JSON)*.
- **Environment variables** — the twelve-factor way and the equivalent of `AddEnvironmentVariables()`; read them with `os.environ.get("APP_PORT", "8080")` *(see Standard Library › System & CLI › 002 · Environment Variables and Platform Info)*.

A common pattern is file defaults overridden by environment variables:

In [ ]:
import os

config = configparser.ConfigParser()
config.read_string("[server]\nport = 8080\n")

os.environ["SHOP_SERVER_PORT"] = "9090"        # simulated deployment setting
port = int(os.environ.get("SHOP_SERVER_PORT", config["server"]["port"]))
print(port)                                     # 9090 — env var wins
del os.environ["SHOP_SERVER_PORT"]

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Load settings | `AddJsonFile` / `AddIniFile` | `config.read(path, encoding="utf-8")` |
| Get a section | `GetSection("Server")` | `config["server"]` |
| Get a string | `config["Server:Host"]` | `config["server"]["host"]` |
| Typed with default | `GetValue<int>("Server:Port", 80)` | `config.getint("server", "port", fallback=80)` |
| Boolean | `GetValue<bool>(...)` | `getboolean(...)` (`yes/no/on/off/true/false/1/0`) |
| Shared defaults | — | `[DEFAULT]` section |
| Placeholders | — | `%(key)s` / `${section:key}` interpolation |
| Bind to object | `IOptions<T>` | build a `dataclass` from a section |
| Save | — | `config.write(f)` |

## Exercises

1. Parse the INI below and print `max_connections` as an `int`, `ssl` as a `bool`, and `pool_timeout` with a fallback of `30.0` (it's missing):
   ```ini
   [database]
   max_connections = 20
   ssl = on
   ```
2. Use `ExtendedInterpolation` so that `[paths] reports = ${base:dir}/reports` resolves with `[base] dir = /data`. Print the resolved value.
3. Build a config with sections `alpha` and `beta` (each with a `weight` key), write it to a temp file, read it back with a new parser, and print the sum of the weights.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import configparser
import tempfile
from pathlib import Path

# 1 — typed getters with a fallback
cfg = configparser.ConfigParser()
cfg.read_string("[database]\nmax_connections = 20\nssl = on\n")
print(cfg.getint("database", "max_connections"),
      cfg.getboolean("database", "ssl"),
      cfg.getfloat("database", "pool_timeout", fallback=30.0))

# 2 — ${section:key}
cfg = configparser.ConfigParser(interpolation=configparser.ExtendedInterpolation())
cfg.read_string("[base]\ndir = /data\n\n[paths]\nreports = ${base:dir}/reports\n")
print(cfg["paths"]["reports"])   # /data/reports

# 3 — write, then read back
cfg = configparser.ConfigParser()
cfg["alpha"] = {"weight": "3"}
cfg["beta"] = {"weight": "4"}
with tempfile.TemporaryDirectory() as d:
    path = Path(d) / "weights.ini"
    with open(path, "w", encoding="utf-8") as f:
        cfg.write(f)
    again = configparser.ConfigParser()
    again.read(path, encoding="utf-8")
    print(sum(again.getint(s, "weight") for s in again.sections()))   # 7
```

</details>